In [1]:
# ============================================================================
# SECTION 1: Installation & Imports
# ============================================================================

# First, install the package (uncomment if not installed):
# !pip install ev2gym
import sys

sys.path.insert(0, "..")

try:
    from ev2gym.models.ev2gym_env import EV2Gym
    from ev2gym.baselines.heuristics import ChargeAsFastAsPossible
    import numpy as np
    print("✓ All imports successful!")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("Please install ev2gym: pip install ev2gym")
    exit()

/home/marinevas/master/tasi/EV2Gym/tutorials/../ev2gym/utilities/loaders.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


✓ All imports successful!


In [2]:
# ============================================================================
# SECTION 2: Understanding the Configuration
# ============================================================================

# EV2Gym uses YAML configuration files to set up simulations
# Let's use a simple configuration for V2G profit maximization

config_file = "../ev2gym/example_config_files/PublicPST.yaml"

print(f"\nUsing config file: {config_file}")
print("\nKey configuration parameters:")
print("  - timescale: 15 minutes per step")
print("  - simulation_length: 96 steps (24 hours)")
print("  - v2g_enabled: True (EVs can discharge back to grid)")
print("  - number_of_charging_stations: 150")
print("  - scenario: public (public charging station behavior)")


Using config file: ../ev2gym/example_config_files/PublicPST.yaml

Key configuration parameters:
  - timescale: 15 minutes per step
  - simulation_length: 96 steps (24 hours)
  - v2g_enabled: True (EVs can discharge back to grid)
  - number_of_charging_stations: 150
  - scenario: public (public charging station behavior)


In [3]:
# ============================================================================
# SECTION 3: Initialize the Environment
# ============================================================================

try:
    env = EV2Gym(
        config_file=config_file,
        save_replay=False,  # Don't save replay for now
        save_plots=False    # Don't generate plots for now
    )
    print("✓ Environment initialized successfully!")
    
    # Reset to get initial state
    state, info = env.reset()
    
    print(f"\nEnvironment Details:")
    print(f"  - Simulation length: {env.simulation_length} steps")
    print(f"  - Time per step: {env.timescale} minutes")
    print(f"  - Total simulation time: {env.simulation_length * env.timescale / 60:.1f} hours")
    print(f"  - Number of transformers: {len(env.transformers)}")
    print(f"  - Number of charging stations: {len(env.charging_stations)}")
    print(f"  - Reward function: {env.reward_function}")
    
except Exception as e:
    print(f"✗ Error initializing environment: {e}")
    exit()


✓ Environment initialized successfully!

Environment Details:
  - Simulation length: 112 steps
  - Time per step: 15 minutes
  - Total simulation time: 28.0 hours
  - Number of transformers: 1
  - Number of charging stations: 9
  - Reward function: <function SquaredTrackingErrorReward at 0x7fded19b6ca0>


In [4]:
import sys
sys.path.append("/home/marinevas/master/tasi/EV2Gym/ev2gym/rl_agent")

from reward import PeakPenaltyReward

In [5]:
# ============================================================================
# SECTION 7: Running a Simple Simulation with Random Actions
# ============================================================================

env.reset()
total_reward = 0
num_steps = 50  # Run for 20 steps (5 hours)
env.set_reward_function(PeakPenaltyReward)

for t in range(112):
    # Get currently connected EVs
    connected_evs = env.current_evs_parked

    N_MAX = env.action_space.shape[0]

    actions = np.zeros(N_MAX)
    
    # Create random actions for each connected EV
    if connected_evs > 0:
        # Random actions between -10 kW (discharge) and 10 kW (charge)
        actions[:connected_evs] = np.random.uniform(-10, 10, size=connected_evs)

    else:
        actions = np.zeros(env.action_space.shape)

    if env.current_step == 112: break

    # Take action
    state, reward, done, truncated, info = env.step(actions)
    
    total_reward += reward
    total_power = np.sum(actions) if len(actions) > 0 else 0
    
    # Print progress every 5 steps
    # if t % 5 == 0:
    #     print(f"{t:4d} | {connected_evs:13d} | Random | {reward:7.2f} | {total_power:7.2f} kW")
    
    if done or truncated:
        break

print("-" * 60)
print(f"Total reward: {total_reward:.2f}")
print("\nNote: Random actions give poor performance!")
print("A good agent should learn better strategies.")

------------------------------------------------------------
Total reward: 577.16

Note: Random actions give poor performance!
A good agent should learn better strategies.


In [6]:
# ============================================================================
# SECTION 8: Running with a Heuristic Baseline
# ============================================================================

env.reset()
agent = ChargeAsFastAsPossible()
total_reward = 0

print("\nRunning simulation with 'Charge As Fast As Possible' heuristic...")
print("\nStep | Connected EVs | Reward | Cum. Reward")
print("-" * 50)

for t in range(num_steps):
    # Get action from heuristic agent
    actions = agent.get_action(env)
    
    # Take action
    state, reward, done, truncated, info = env.step(actions)
    
    total_reward += reward
    n_evs = env.current_evs_parked
    
    if t % 5 == 0:
        print(f"{t:4d} | {n_evs:13d} | {reward:7.2f} | {total_reward:10.2f}")
    
    if done or truncated:
        break

print("-" * 50)
print(f"Final total reward: {total_reward:.2f}")


Running simulation with 'Charge As Fast As Possible' heuristic...

Step | Connected EVs | Reward | Cum. Reward
--------------------------------------------------
   0 |             0 |    0.00 |       0.00
   5 |             0 |    0.00 |       0.00
  10 |             0 |    0.00 |       0.00
  15 |             2 | 2161.50 |    2161.50
  20 |             2 | 1167.17 |   10532.04
  25 |             2 |    8.28 |   12347.36
  30 |             2 |    0.00 |   12347.27
  35 |             3 |  363.37 |   13441.05
  40 |             3 | 1065.97 |   17517.25
  45 |             2 |   75.98 |   20467.24
--------------------------------------------------
Final total reward: 20498.16


In [7]:
# ============================================================================
# SECTION 9: Understanding Rewards
# ============================================================================

print("\nThe reward signal depends on the reward function used.")
print("Common reward components:")
print("  1. Economic: Profit from energy arbitrage")
print("     - Revenue from discharging during high prices")
print("     - Cost of charging during low prices")
print("  2. User satisfaction: Meeting EV charging needs")
print("     - Penalty if EV leaves without desired energy")
print("  3. Grid constraints: Respecting transformer limits")
print("     - Penalty for overloading transformers")
print("  4. Power tracking: Following setpoint profiles")
print("     - Penalty for deviating from target power")


The reward signal depends on the reward function used.
Common reward components:
  1. Economic: Profit from energy arbitrage
     - Revenue from discharging during high prices
     - Cost of charging during low prices
  2. User satisfaction: Meeting EV charging needs
     - Penalty if EV leaves without desired energy
  3. Grid constraints: Respecting transformer limits
     - Penalty for overloading transformers
  4. Power tracking: Following setpoint profiles
     - Penalty for deviating from target power


In [8]:
# ============================================================================
# SECTION 10: Examining Electricity Prices
# ============================================================================

env.reset()

if hasattr(env, 'charge_prices') and env.charge_prices is not None:
    print("\nElectricity prices (first 24 steps / 6 hours):")
    print("\nStep | Time  | Buy Price | Sell Price")
    print("-" * 45)
    
    for t in range(min(24, len(env.charge_prices))):
        hour = (env.sim_date.hour + t * env.timescale / 60) % 24
        buy_price = env.charge_prices[0][t] # env.charge prices vem como um array de arrays (1 por charging station) de precos a cada 15min nesse dia
        sell_price = env.discharge_prices[0][t] if hasattr(env, 'discharge_prices') else buy_price
        
        if t % 4 == 0:  # Every hour
            print(f"{t:4d} | {hour:5.1f} | {buy_price:9.4f} | {sell_price:10.4f}")
    
    print("\nKey insight: An optimal strategy charges when prices are LOW")
    print("and discharges when prices are HIGH!")
else:
    print("\nPrice information not directly accessible in this configuration.")


Electricity prices (first 24 steps / 6 hours):

Step | Time  | Buy Price | Sell Price
---------------------------------------------
   0 |   5.0 |   -0.1672 |     0.1672
   4 |   6.0 |   -0.2166 |     0.2166
   8 |   7.0 |   -0.2247 |     0.2247

Key insight: An optimal strategy charges when prices are LOW
and discharges when prices are HIGH!


In [9]:
# ============================================================================
# SECTION 11: Understanding Transformer Constraints
# ============================================================================

env.reset()

print(f"\nNumber of transformers: {len(env.transformers)}")

for i, transformer in enumerate(env.transformers):
    print(f"\nTransformer {i}:")
    print(f"  - Max power: {transformer.max_power[0]} kW")
    print(f"  - Connected charging stations: {len(transformer.cs_ids)}")
    print(f"  - Current load: {transformer.inflexible_load[0]:.1f} kW")
    print(f"  - Remaining capacity: {transformer.max_power[0] - transformer.inflexible_load[0]:.1f} kW")

print("\nImportant: The agent must respect transformer capacity limits!")
print("Overloading transformers typically incurs heavy penalties.")


Number of transformers: 1

Transformer 0:
  - Max power: 100.0 kW
  - Connected charging stations: 9
  - Current load: 0.0 kW
  - Remaining capacity: 100.0 kW

Important: The agent must respect transformer capacity limits!
Overloading transformers typically incurs heavy penalties.


In [10]:
# ============================================================================
# SECTION 12: Testing Complete Episode
# ============================================================================

print("\nRunning a complete episode (full 24 hours)...")

env.reset()
agent = ChargeAsFastAsPossible()
total_reward = 0
episode_stats = []

for t in range(env.simulation_length):
    actions = agent.get_action(env)
    state, reward, done, truncated, info = env.step(actions)
    
    total_reward += reward
    episode_stats.append({
        'step': t,
        'connected_evs': env.current_evs_parked,
        'reward': reward,
        'total_power': np.sum(actions) if len(actions) > 0 else 0
    })
    
    if done or truncated:
        break

print(f"\nEpisode completed!")
print(f"  - Total steps: {len(episode_stats)}")
print(f"  - Total reward: {total_reward:.2f}")
print(f"  - Average reward per step: {total_reward / len(episode_stats):.2f}")
print(f"  - Max EVs connected: {max(s['connected_evs'] for s in episode_stats)}")
print(f"  - Average EVs connected: {np.mean([s['connected_evs'] for s in episode_stats]):.1f}")


Running a complete episode (full 24 hours)...

Episode completed!
  - Total steps: 112
  - Total reward: 14095.34
  - Average reward per step: 125.85
  - Max EVs connected: 4
  - Average EVs connected: 0.7


In [11]:
# ============================================================================
# SECTION 13: Summary & Next Steps
# ============================================================================

print("""
Summary:
--------
✓ Successfully initialized EV2Gym environment
✓ Understood state representation (numpy array)
✓ Understood action space (continuous power values per EV)
✓ Tested with random actions (poor performance)
✓ Tested with heuristic baseline (better performance)
✓ Explored electricity prices and transformer constraints
✓ Ran complete episode simulation

Key Takeaways:
--------------
1. EVs arrive and depart dynamically throughout the day
2. Actions are charging/discharging power for each connected EV
3. Rewards depend on: profits, user satisfaction, grid constraints
4. Transformer capacity must be respected
5. Electricity prices vary (day-ahead market prices)

Next Steps for RL Implementation:
----------------------------------
1. Design state function (aggregate EV info + prices + time)
2. Discretize action space for DQN
3. Design reward function (balance multiple objectives)
4. Implement DQN agent with replay buffer
5. Train and evaluate against baselines

Ready to implement your RL agent! 🚀
""")


Summary:
--------
✓ Successfully initialized EV2Gym environment
✓ Understood state representation (numpy array)
✓ Understood action space (continuous power values per EV)
✓ Tested with random actions (poor performance)
✓ Tested with heuristic baseline (better performance)
✓ Explored electricity prices and transformer constraints
✓ Ran complete episode simulation

Key Takeaways:
--------------
1. EVs arrive and depart dynamically throughout the day
2. Actions are charging/discharging power for each connected EV
3. Rewards depend on: profits, user satisfaction, grid constraints
4. Transformer capacity must be respected
5. Electricity prices vary (day-ahead market prices)

Next Steps for RL Implementation:
----------------------------------
1. Design state function (aggregate EV info + prices + time)
2. Discretize action space for DQN
3. Design reward function (balance multiple objectives)
4. Implement DQN agent with replay buffer
5. Train and evaluate against baselines

Ready to impleme